# LAC News Monitor

In [ ]:
import os
import json
import feedparser
import pandas as pd
from google import genai
from arcgis.gis import GIS
from arcgis.features import FeatureLayerCollection


In [ ]:
#ArcGIS Login Details
ORG_URL = "https://idb-gis.maps.arcgis.com"
CLIENT_ID = "xunIRcqrmgb4gB0T"
CLIENT_SECRET = "c927d97d667c40c0b7f07350943b475e"
LAYER_ID = "dfbb6dbc3a384d8aa436803b896eacd6"

YOUR_GOOGLE_NEWS_RSS_URL = 'https://news.google.com/rss/search?q=(Drought+OR+Sequía+OR+Seca+OR+Earthquake+OR+Terremoto+OR+Sismo+OR+Flooding+OR+Inundación+OR+Alagamento+OR+"Extreme+Cold"+OR+"Frío+Extremo"+OR+"Frio+Extremo"+OR+"Extreme+Heat"+OR+"Calor+Extremo"+OR+Cyclones+OR+Ciclón+OR+Ciclone+OR+Landslides+OR+Deslave+OR+Derrumbe+OR+Deslizamento+OR+Storms+OR+Tormenta+OR+Tempestade+OR+Tsunami+OR+Wildfires+OR+Incendio+OR+Incêndio+OR+Tornado+OR+"Volcanic+Eruption"+OR+"Erupción+Volcánica"+OR+"Erupção+Volcânica")+(Argentina+OR+Bahamas+OR+Barbados+OR+Belize+OR+Belice+OR+Bolivia+OR+Brazil+OR+Brasil+OR+Chile+OR+Colombia+OR+"Costa+Rica"+OR+"Dominican+Republic"+OR+"República+Dominicana"+OR+Ecuador+OR+"El+Salvador"+OR+Guatemala+OR+Guyana+OR+Haiti+OR+Haití+OR+Honduras+OR+Jamaica+OR+Mexico+OR+México+OR+Nicaragua+OR+Panama+OR+Panamá+OR+Paraguay+OR+Peru+OR+Perú+OR+Suriname+OR+Surinam+OR+"Trinidad+and+Tobago"+OR+"Trinidad+y+Tobago"+OR+Uruguay+OR+Venezuela)+when:7d&hl=en-US&gl=US&ceid=US:en' 

In [ ]:

# 1. Initialize Gemini Client & Fetch Feed
client = genai.Client(api_key= os.getenv("GEMINI_API_KEY"))
feed = feedparser.parse("YOUR_GOOGLE_NEWS_RSS_URL")

parsed_articles = []

# 2. Analyze with Gemini Agent
for entry in feed.entries:
    prompt = f"""
    Analyze this disaster news item in Latin America and the Caribbean.
    Title: {entry.get('title', '')}
    Summary: {entry.get('summary', '')}
    Link: {entry.get('link', '')}

    Extract country names (no coordinates needed) and return ONLY a raw JSON object:
    {{
      "valid": true/false (true ONLY if active natural disaster in a Latin American or Caribbean country),
      "country": "Primary IDB country name in English",
      "disaster_type": "Drought/Earthquake/Flooding/Extreme Cold/Extreme Heat/Cyclones/Landslides/Storms/Tsunami/Wildfires/Tornado/Volcanic Eruptions",
      "headline_en": "Translated headline in English",
      "summary_en": "Translated summary in English without HTML tags",
      "date_reported": "{entry.get('published', '')}",
      "source_url": "{entry.get('link', '')}"
    }}
    """
    
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(response_mime_type="application/json")
        )
        data = json.loads(response.text)
        
        if data.get("valid"):
            data.pop("valid", None)  # Remove 'valid' key so it doesn't break AGOL schema
            parsed_articles.append(data)
    except Exception:
        continue
        
    time.sleep(6)  # Pause to respect Gemini free-tier rate limits (~10 RPM)

# 3. Process DataFrame & Clean Dates
df = pd.DataFrame(parsed_articles)

if not df.empty and "date_reported" in df.columns:
    df["date_reported"] = pd.to_datetime(df["date_reported"], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')

csv_file = "disasters_lac.csv"
df.to_csv(csv_file, index=False)

gis = GIS("https://www.arcgis.com", client_id=CLIENT_ID, client_secret=CLIENT_SECRET)
item = gis.content.get(LAYER_ID)
flc = FeatureLayerCollection.fromitem(item)
flc.manager.overwrite(csv_file)